# Calibrar las recomendaciones con tus datos

Este notebook **no escribe nada**. Prueba configuraciones y mide cuál acierta más, para que después
pegues la ganadora en `rec_oracle.py`.

Cómo mide: entrena con la historia hasta hace `DIAS_BACKTEST` días y mira **qué ítems nuevos compró
después cada entidad**. Lo compara siempre contra la línea de base, que es "ofrecer lo más vendido
del segmento".

| Qué elegir | Cómo se decide |
|---|---|
| **A qué nivel recomendar** (SKU, submarca, familia, marca) | el que más acierta. Si el cliente medio compra 2 ítems de 5.000, el nivel es demasiado fino |
| **Los mínimos** (`min_soporte`, `min_penetracion`) | los que más aciertan sin quedarse sin candidatos |
| **Los parámetros** (`k_vecinos`, `k_factores`, `k_clusters`) | igual, por backtest |
| **Canasta o repertorio** | canasta mira lo que se compra el mismo día; repertorio, todo lo del año. Lo decide el backtest |
| **Qué algoritmo** | ese no hay que elegirlo: la corrida diaria ya lo elige por segmento |

Corré esto cuando cambie el negocio o la fuente; no hace falta todos los días.

In [ ]:
# Parámetros
import json, os

FECHA_EJECUCION = os.getenv("RC_FECHA_EJECUCION") or None

# Niveles de ítem a probar: cada uno es la lista de columnas de ese nivel. Tienen que venir
# en SQL_FUENTE. Del más fino al más grueso. (RC_NIVELES los pisa, en JSON.)
NIVELES_ITEM = json.loads(os.getenv("RC_NIVELES") or "null") or [
    ["SK_PRODUCTO", "BD_PRODUCTO"],
    ["BK_SUBMARCA", "BD_SUBMARCA"],
    ["BK_FAMILIA", "BD_FAMILIA"],
]

# Qué parámetros barrer. Cuantos más, más tarda: son todas las combinaciones.
REJILLA = json.loads(os.getenv("RC_REJILLA") or "null") or {
    "afinidad": ["canasta", "repertorio"],   # canasta = lo que se compra el mismo día
    "min_penetracion": [0.01, 0.02, 0.05],
    "min_soporte": [5, 20],
}
print(f"{len(NIVELES_ITEM)} niveles, rejilla {REJILLA}")

## 1. Leer la fuente una sola vez

In [ ]:
import sys, time
sys.path.insert(0, os.getcwd())
import pandas as pd
import rec_oracle as io
from rec_engine import explorar, bloque_config

log = io.configurar_logging("calibracion")
cfg = io.build_config(fecha_ejecucion=FECHA_EJECUCION)
desde, hasta = io.ventana(cfg)
log.info("ventana: %s a %s", desde.date(), hasta.date())

with io.conexion_origen() as conn:
    fuente = io.leer_fuente(conn, cfg)
print(f"{len(fuente):,} filas")
fuente.head(3)

## 2. Probar todas las configuraciones

Ojo con el tiempo: son `niveles x combinaciones` corridas de backtest. Arrancá con pocas.

In [ ]:
t0 = time.time()
tabla = explorar(fuente, cfg, niveles_item=NIVELES_ITEM, rejilla=REJILLA)
print(f"{len(tabla)} configuraciones en {time.time() - t0:.0f}s")
tabla

## 3. Cómo leer la tabla

| Columna | Qué mirar |
|---|---|
| `items_por_entidad` | menos de 3 y el nivel es demasiado fino: no hay con qué comparar |
| `densidad` | qué tan llena está la matriz. Muy baja (menos de 0,5%) = vecindarios vacíos |
| `precision` | de lo recomendado, qué fracción se compró después |
| `precision_popularidad` | lo mismo ofreciendo lo más vendido del segmento |
| `mejora_vs_popularidad` | **el número que importa**. Si no llega a 1,2 en ningún lado, no vale la pena personalizar en ese nivel |
| `usd_acertado` | cuánta plata había en lo que acertó |
| `afinidad` | `canasta` suele ganar cuando hay complementos reales; `repertorio` cuando la compra es dispersa |
| `algoritmos_elegidos` | qué ganó en cada segmento |

Si todas las filas dan precisión 0: en el tramo evaluado nadie compró ítems nuevos. Probá un nivel
más grueso, o subí `DIAS_BACKTEST` para mirar un tramo más largo.

In [ ]:
cols = ["nivel_item", "items", "items_por_entidad", "densidad", "precision",
        "precision_popularidad", "mejora_vs_popularidad", "usd_acertado", "segundos"]
tabla[cols].head(15)

## 4. El bloque para pegar en `rec_oracle.py`

In [ ]:
mejor = tabla.iloc[0]
print(bloque_config(mejor, cfg))

Pegá eso en `rec_oracle.py`, en la sección 2 y 3, y corré `run_recomendaciones.ipynb`.

**No copies ciegamente la primera fila.** Si la segunda tiene casi la misma precisión con un nivel más
grueso, quedate con la más gruesa: son menos filas, más estable en el tiempo y más fácil de accionar
para el vendedor.